# EP/WXT 指向观测值班演示 / Pointing observation walkthrough

默认使用本机 EP260809a 的 `06800001692_32` 官方 L2/L3 产品。按顺序运行；在人工核查区域前不要批准。其他观测通过 `JINWU_WXT_OBS_ROOT` 指定目录，并修改源坐标和 `source_id`。产物写入新的独立工作区。

The default uses local official EP260809a L2/L3 products. Run cells in order. Set `JINWU_WXT_OBS_ROOT` for another observation and update the source coordinates and `source_id`. Outputs go to a new workspace.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace
import json, os
from IPython.display import display, Image
from astropy.io import fits
from jinwu.core.config import instrument
from jinwu.ep.wxt import WXTPointingInput, WXTPointingPipeline

anchor = Path.cwd().resolve()
repo = next((p for p in (anchor, *anchor.parents) if (p / "packages/jinwu-ep").is_dir()), None)
if repo is None:
    raise RuntimeError("Open this Notebook from inside the JinWu checkout / 请在仓库内打开 Notebook")
default_root = repo / "test/EP260809adata/EP260809a/06800001692_32"
root = Path(os.environ.get("JINWU_WXT_OBS_ROOT", default_root)).expanduser().resolve()
if not root.is_dir():
    raise FileNotFoundError(f"WXT observation directory not found: {root}")
output_base = Path(os.environ.get("JINWU_EXAMPLE_OUTPUT_ROOT", repo / "examples/_outputs")).expanduser().resolve()
output_root = output_base / datetime.now(timezone.utc).strftime("wxt_%Y%m%dT%H%M%S%fZ")
print("Input:", root)
print("Output:", output_root)
print("Files:", len(list(root.iterdir())))

## 1. 输入与配置 / Input and configuration

确认这是一个正常指向观测目录，并根据源表确认 RA、Dec 与 `source_id`。默认关闭自动批准区域。

Confirm the observation and source identity. Automatic region approval is disabled.

In [ ]:
target_id, ra_deg, dec_deg, source_id = "EP260809a", 328.573, 17.607, "s1"  # Replace for another target
inp = WXTPointingInput(target_id=target_id, root=root, output_root=output_root, source_id=source_id, ra_deg=ra_deg, dec_deg=dec_deg, auto_approve_regions=False)
cfg = instrument("WXT")
cfg.reporting = replace(cfg.reporting, print_summary=False)
pipe = WXTPointingPipeline(inp, config=cfg)
pipe.validate_input()
print("Observation:", source_id, "RA/Dec:", ra_deg, dec_deg)
print("Workspace:", pipe.workspace)

## 2. 运行到区域检查 / Run through region QC

`needs_review` 是预期暂停状态。该阶段生成源区、背景区、ARM 排除区、临时事件和曝光诊断。

`needs_review` is the expected pause for source, background, ARM and exposure inspection.

In [ ]:
preview = pipe.run(until="exposure_arm_qc", resume=False)
print("Status:", preview.status)
print("Workspace:", preview.workspace)
assert preview.status == "needs_review", "Inspect the stage log before continuing"

## 3. 人工核查 / Manual review

逐项查看区域文件和曝光诊断。在 DS9 中以观测图像为底图叠加 `source`、`background_effective` 和 `arm` 区域，确认位置、PSF 翼、邻源与无曝光区域；`alpha` 与覆盖率必须有限且合理。若不合适，停止并修改输入或区域。

Overlay the listed regions on the observation image in DS9. Check source placement, PSF wings, neighbours, excluded ARM area and zero-exposure pixels. Stop if the geometry or exposure scaling is questionable.

In [ ]:
region_dir = pipe.workspace / "regions"
regions = json.loads((region_dir / "regions.json").read_text(encoding="utf-8"))
qc = json.loads((region_dir / "exposure_qc.json").read_text(encoding="utf-8"))
for label in ("source", "background", "background_effective", "arm"):
    print(f"{label}: {regions.get(label)}")
print("alpha:", qc.get("alpha"))
for label in ("source", "background", "warning", "exposure_coverage_warning", "boundary_clipping_warning", "region_geometry_warning"):
    if label in qc:
        print(label, qc[label])
preview_image = next(iter(sorted(root.glob("*img.gif"))), None)
if preview_image:
    display(Image(filename=str(preview_image)))
print("Open the region files on the FITS image in DS9 before approval.")

In [ ]:
# Gross geometry overlay for teaching; DS9 remains the review tool for ARM exclusions.
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Polygon, Circle
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
image_path = next(iter(sorted(root.glob("*.img"))), None)
if image_path is not None:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with fits.open(image_path) as hdus:
            image_hdu = next(h for h in hdus if getattr(h.data, "ndim", 0) == 2)
            image_data = np.asarray(image_hdu.data, dtype=float)
            header = image_hdu.header
            wcs = WCS(header).celestial
            cx, cy = wcs.all_world2pix([[ra_deg, dec_deg]], 0)[0]
    radius_deg = cfg.regions.source_radius_arcsec / 3600
    radius_px = radius_deg / float(np.mean(np.abs(proj_plane_pixel_scales(wcs))))
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image_data, origin="lower", cmap="gray", norm=LogNorm(vmin=1, vmax=max(2, float(np.nanmax(image_data)))))
    ax.add_patch(Circle((cx, cy), radius_px, fill=False, edgecolor="lime", lw=2, label="source"))
    for line in Path(regions["background"]).read_text().splitlines():
        if not line.startswith("polygon("):
            continue  # excludes are not drawn in this gross-geometry preview
        vertices = np.fromstring(line[8:].split(")")[0], sep=",").reshape(-1, 2)
        vertices[:, 0] = vertices[:, 0] * header["LTM1_1"] + header["LTV1"] - 1
        vertices[:, 1] = vertices[:, 1] * header["LTM2_2"] + header["LTV2"] - 1
        ax.add_patch(Polygon(vertices, fill=False, edgecolor="orange", lw=1))
    ax.set_xlim(cx - 80, cx + 80)
    ax.set_ylim(cy - 80, cy + 80)
    ax.set_title("Source (green) and background sectors (orange); inspect ARM in DS9")
    plt.show()
else:
    print("No mission image found; inspect the FITS exposure map and regions in DS9.")

## 4. 明确批准并续跑 / Explicit approval and resume

只有完成上面的人工核查，才在提示中输入 `APPROVE`。不要用“全部运行”跳过阅读。

Type `APPROVE` only after manual inspection.

In [ ]:
decision = input("After reviewing source/background/ARM regions, type APPROVE: ").strip()
if decision != "APPROVE":
    raise RuntimeError("Region approval withheld; downstream analysis did not run")
pipe.approve_regions(note="reviewed interactively during WXT teaching demonstration")
result = pipe.run(resume=True)
print("Status:", result.status, "workspace:", result.workspace)
print("alpha:", result.alpha, "T90 source PHA:", result.t90_source_pha)
print("report:", result.report)

## 5. 产物与诊断 / Products and diagnostics

检查 PHA 的 BACKFILE、RESPFILE、ANCRFILE、EXPOSURE、BACKSCAL，以及持续时间、拟合残差和光变曲线。完成状态仅说明流程结束，不能替代科学审核。

Inspect PHA links, exposure and BACKSCAL, plus durations, fit residuals and light curves. A completed pipeline status is not a scientific validation.

In [ ]:
print("Duration:", result.duration)
print("Segments:", len(result.segments))
for label, path in (("source", result.t90_source_pha), ("background", result.t90_background_pha)):
    if path is None or not path.is_file():
        print(label, "PHA unavailable; inspect pipeline status and logs")
        continue
    with fits.open(path) as hdus:
        h = hdus["SPECTRUM"].header
        print(label, {key: h.get(key) for key in ("EXPOSURE", "BACKSCAL", "BACKFILE", "RESPFILE", "ANCRFILE")})
        for key in ("BACKFILE", "RESPFILE", "ANCRFILE"):
            value = h.get(key)
            if value and value.upper() not in ("NONE", "NULL"):
                print("  ", key, "exists:", (path.parent / value).exists())
print("Report:", result.report, "exists:", bool(result.report and result.report.is_file()))
print("Plots:", list(result.plot_groups))

In [ ]:
result.display()